<a href="https://colab.research.google.com/github/pavithraprem45/projects-/blob/main/Python_cardwar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


import random #to generate random number or random selections , shuffling deck
import json # json dump json load , ability to read and write complex data
import os # handles file manipulation , directory managemnt ,

# 1. Deck and Card Setup Functions

def create_deck():
  # *Creates a standard 52-card deck with one Joker (Power Boost), and assigns
    # a numerical value.
  #*Joker is the highest rank (value: 15) for comparison purposes,
    # but its 'POWER_BOOST' is still +10.

    suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
    ranks = {
        '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10,
        'Jack': 11, 'Queen': 12, 'King': 13, 'Ace': 14
    }

    deck = []
    for suit in suits:
        for rank, value in ranks.items():
            card = {'rank': rank, 'suit': suit, 'value': value}
            deck.append(card)

    # *Joker has the 'POWER_BOOST' effect
    # *The value is set to 0, but the boost makes it 10.
    deck.append({'rank': 'Joker', 'suit': 'Special', 'value': 0, 'effect': 'POWER_BOOST'})

    random.shuffle(deck)
    return deck

def deal_cards(deck):
    # *Splits the shuffled deck equally between two players
    mid_point = len(deck) // 2
    return deck[:mid_point], deck[mid_point:]

#  2. Game Logic Functions

def get_player_info():
   # *Prompts users for names and the maximum number of rounds to play.

    print("\n ---|♠ Welcome to Python Card War! ♠|---")
    name1 = input("🃚 Player 1, enter your name: ").strip() or "Player 1"
    name2 = input("🃚 Player 2, enter your name: ").strip() or "Player 2"

    while True:
        try:
            limit = int(input("Enter the maximum number of rounds to play: "))
            if limit > 0:
                break
            else:
                print("Please enter a number greater than 0.")
        except ValueError:
            print("Invalid input. Please enter a whole number.")

    return name1, name2, limit

def war_challenge(player1_deck, player2_deck, name1, name2, war_cards, scores):

    # *Handles the recursive "War" tie-breaker. Draws 3 cards face down, 1 face
    # up.
    #*This function continues until a winner is found or a player runs out of cards.

    print("\n **❗WAR CHALLENGE INITIATED❗**")

    # *1- Check if players have enough cards (3 face-down + 1 face-up = 4 total)
    if len(player1_deck) < 4:
        print(f"⚡ {name1} does not have enough cards to fight the War. {name2} wins the War by default!")
        return 1, war_cards
    if len(player2_deck) < 4:
        print(f"⚡{name2} does not have enough cards to fight the War. {name1} wins the War by default!")
        return 0, war_cards

    #* 2- Draw 3 cards face down for the War Pile
    print(f"{name1} places 3 cards face down.")
    war_cards.extend([player1_deck.pop(0) for _ in range(3)])

    print(f"{name2} places 3 cards face down.")
    war_cards.extend([player2_deck.pop(0) for _ in range(3)])

    #*3- Draw 1 card face up (The Showdown Card)
    p1_showdown_card = player1_deck.pop(0)
    p2_showdown_card = player2_deck.pop(0)
    war_cards.append(p1_showdown_card)
    war_cards.append(p2_showdown_card)

    print(f"\n---📢 WAR SHOWDOWN 📢---")
    print(f"{name1}'s Showdown Card: {p1_showdown_card['rank']} of {p1_showdown_card['suit']} (Value: {p1_showdown_card['value']})")
    print(f"{name2}'s Showdown Card: {p2_showdown_card['rank']} of {p2_showdown_card['suit']} (Value: {p2_showdown_card['value']})")
    print(f"Total cards at stake: {len(war_cards)}")

    p1_value = p1_showdown_card['value']
    p2_value = p2_showdown_card['value']

    #* Check for the Power Boost effect on the showdown cards (optional, but consistent)
    if p1_showdown_card.get('effect') == 'POWER_BOOST':
        p1_value += 10
        print(f"🃏 **JOKER BOOST**! {name1}'s showdown card value is boosted to {p1_value}.")
    if p2_showdown_card.get('effect') == 'POWER_BOOST':
        p2_value += 10
        print(f"🃏 **JOKER BOOST**! {name2}'s showdown card value is boosted to {p2_value}.")

    # *4- Determine War Winner
    if p1_value > p2_value:
        print(f"👑 {name1} wins the War! Taking all {len(war_cards)} cards.")
        return 0, war_cards # Player 1 wins the war
    elif p2_value > p1_value:
        print(f"👑 {name2} wins the War! Taking all {len(war_cards)} cards.")
        return 1, war_cards # Player 2 wins the war
    else:
        # *if its a Tie again, Initiate another War
        print("⚖️ It's a tie again! Another War is declared!")
        return war_challenge(player1_deck, player2_deck, name1, name2, war_cards, scores)


def compare_cards_and_check_war(card1, card2, player1_deck, player2_deck, name1, name2, scores):
    #* Compares two drawn cards, handles Power Boost and Revenge Card, and initiates War on a rank tie.
    #* Returns the winner, cards won, and updated scores.

    print(f"\n{name1} draws: {card1['rank']} of {card1['suit']} (Value: {card1['value']})")
    print(f"{name2} draws: {card2['rank']} of {card2['suit']} (Value: {card2['value']})")

    # The cards drawn this round are the initial stake for the War
    war_cards = [card1, card2]

    p1_value = card1['value']
    p2_value = card2['value']

    # *CHALLENGE/ SPECIAL CARD -JOKER LOGIC: Power Boost
    if card1.get('effect') == 'POWER_BOOST':
        p1_value += 10
        print(f"🃏 **JOKER BOOST**! {name1}'s card value is increased by 10 (Total: {p1_value}).")

    if card2.get('effect') == 'POWER_BOOST':
        p2_value += 10
        print(f"🃏 **JOKER BOOST**! {name2}'s card value is increased by 10 (Total: {p2_value}).")

    winner = None
    # *0 for P1, 1 for P2, None for tie

    # *Check for standard win condition (Higher value wins)
    if p1_value > p2_value:
        print(f"🏆 {name1} wins the round!")
        winner = 0
    elif p2_value > p1_value:
        print(f"🏆 {name2} wins the round!")
        winner = 1
    else:
        #*WAR CHALLENGE: Triggered by a tie in final value/rank
        #*If values are equal, initiate the War sequence
        winner, cards_won = war_challenge(player1_deck, player2_deck, name1, name2, war_cards, scores)

        # *The winner of the War gets all cards added to their score/deck (handled in play_card_war)
        return winner, cards_won, scores # Return winner, all cards won, and scores

    # *If no War occurred, award the primary point (and check for Revenge Card)

    # * Challenge 2: Revenge Card (Bonus point if King loses to Ace)

    # *P1 lost (winner=1) and P1 had King and P2 had Ace
    if winner == 1 and card1['rank'] == 'King' and card2['rank'] == 'Ace':
        scores[0] += 1
        print(f"👑 **REVENGE CARD**: {name1} drew a King and lost to an Ace! {name1} earns a **bonus point**!")
    # *P2 lost (winner=0) and P2 had King and P1 had Ace
    elif winner == 0 and card2['rank'] == 'King' and card1['rank'] == 'Ace':
        scores[1] += 1
        print(f"👑 **REVENGE CARD**: {name2} drew a King and lost to an Ace! {name2} earns a **bonus point**!")

    #*Standard round win: 1 point + the two cards won for the War pile (war_cards will only contain 2 cards)
    return winner, war_cards, scores

def announce_game_winner(scores, name1, name2, max_rounds):
    """
    Prints the final scores and announces the overall winner of the game.
    """
    print(f"\n--- 🏁 Game Over! ---")
    print(f"Total points won: {name1}: {scores[0]} | {name2}: {scores[1]}")
    # *Also showing the number of cards remaining
    print("The goal of War is to win all the cards! Points are secondary.")

    if scores[0] > scores[1]:
        print(f"🎉 Congratulations, 🥇*{name1}*🥇 is the overall game winner based on points!")
    elif scores[1] > scores[0]:
        print(f"🎉 Congratulations, 🥇*{name2}*🥇 is the overall game winner based on points!")
    else:
        print("🤝 The game ends in a tie!")

# *3. Save and Load Functions
# ... (Save and Load functions are kept simple to maintain file structure, but deck saving is now crucial)
# NOTE: In a traditional War game, the winner's "deck" is actually a discard pile that is shuffled and used when the main deck runs out.
# For simplicity, we continue to treat the player's deck as their draw pile, which also acts as their win/reserve pile.

SAVE_FILE = "cardwar_save.json"

def save_game(name1, name2, max_rounds, player1_deck, player2_deck, scores, round_count, extra_turn_player):
    """
    Saves the complete game state to a JSON file.
    """
    game_state = {
        'name1': name1,
        'name2': name2,
        'max_rounds': max_rounds,
        'player1_deck': player1_deck,
        'player2_deck': player2_deck,
        'scores': scores,
        'round_count': round_count,
        'extra_turn_player': extra_turn_player
    }
    try:
        with open(SAVE_FILE, 'w') as f:
            json.dump(game_state, f, indent=4)
        print(f"\n✅ Game successfully saved to '{SAVE_FILE}'!")
    except IOError as e:
        print(f"\n❌ Error saving game: {e}")

def load_game():
    """
    Loads a saved game state from the JSON file, if one exists.
    """
    if not os.path.exists(SAVE_FILE):
        return None

    try:
        with open(SAVE_FILE, 'r') as f:
            game_state = json.load(f)
        print(f"\n✅ Game successfully loaded from '{SAVE_FILE}'!")
        return game_state
    except (IOError, json.JSONDecodeError) as e:
        print(f"\n❌ Error loading game: {e}. Starting new game.")
        return None

# * 4. Main Game Function

def play_card_war():
    """
    The main game loop, managing initialization, round progression, and War handling.
    """
    name1, name2, max_rounds = "Player 1", "Player 2", 0
    scores = [0, 0]
    round_count = 0
    extra_turn_player = None
    player1_deck = []
    player2_deck = []

    # --- Game Setup: Load or Start New ---
    loaded_state = None
    if os.path.exists(SAVE_FILE):
        load_choice = input("A saved game exists. Load it? (y/n): ").lower()
        if load_choice == 'y':
            loaded_state = load_game()
            if loaded_state:
                name1 = loaded_state['name1']
                name2 = loaded_state['name2']
                max_rounds = loaded_state['max_rounds']
                player1_deck = loaded_state['player1_deck']
                player2_deck = loaded_state['player2_deck']
                scores = loaded_state['scores']
                round_count = loaded_state['round_count']
                extra_turn_player = loaded_state['extra_turn_player']

                print(f"Resuming game for {name1} vs {name2} (Round {round_count + 1} of {max_rounds})")

    # *If starting new game
    if not player1_deck:
        name1, name2, max_rounds = get_player_info()
        full_deck = create_deck()
        player1_deck, player2_deck = deal_cards(full_deck)

    #*5 Main Game Loop: Runs until max_rounds is reached
    if round_count < max_rounds:
        while round_count < max_rounds:
            round_count += 1
            round_header = f"\n--- Round {round_count} ---"
            print(round_header)

            # *Check if players have cards before starting round
            if not player1_deck:
                print(f"Out of cards! **{name1}** has no cards left.")
                break
            if not player2_deck:
                print(f"Out of cards! **{name2}** has no cards left.")
                break

            # *Draw Cards with sequential interaction
            input(f"{name1}, press **Enter** to draw your card: ")
            card1 = player1_deck.pop(0)
            print(f"✅ {name1} has drawn their card!")

            input(f"{name2}, press **Enter** to draw your card: ")
            card2 = player2_deck.pop(0)
            print(f"✅ {name2} has drawn their card!")

            # *Determine Winner and Handle War
            # *The function now returns the winner, all the cards won (2 or more), and the updated scores
            winner, cards_won, scores = compare_cards_and_check_war(
                card1, card2, player1_deck, player2_deck, name1, name2, scores
            )

            # *Award the primary point for winning the round (if a standard win occurred)
            # *and collect the cards won into the winner's deck/pile
            if winner == 0:
                scores[0] += 1
                player1_deck.extend(cards_won)
            elif winner == 1:
                scores[1] += 1
                player2_deck.extend(cards_won)

            # *Display current scores and card counts
            print(f"\nCurrent Score: {name1}: {scores[0]} | {name2}: {scores[1]}")
            print(f"Cards Remaining: {name1}: {len(player1_deck)} | {name2}: {len(player2_deck)}")

            # * check to end if all cards are concentrated in one player's hands
            if len(player1_deck) == 0 or len(player2_deck) == 0:
                print("\n 🔴One player has captured all the cards! Game over.")
                break


    # *6.End Game Processing
    announce_game_winner(scores, name1, name2, max_rounds)

    final_save_choice = input("\nGame finished! Type **'s'** to save the final results, or press **Enter** to exit: ").lower()

    if final_save_choice == 's':
        save_game(name1, name2, max_rounds, player1_deck, player2_deck, scores, round_count, extra_turn_player)

    elif os.path.exists(SAVE_FILE):
        os.remove(SAVE_FILE)
        print(f"\n **Save file '{SAVE_FILE}' cleaned up.**")

    print("\n||Thank you for playing!||")

# Execute the game by calling the main function
if __name__ == "__main__": # is the file being run as the main program
    play_card_war()


 ---|♠ Welcome to Python Card War! ♠|---


KeyboardInterrupt: Interrupted by user